# Disease diagnosis RAG — end-to-end walkthrough

Educational pipeline: **ingest → retrieve → rerank** (BM25, k-NN, hybrid + RRF, cross-encoder rerank).

## Prerequisites

1. `.env` at project root with `OPENSEARCH_*` credentials
2. OpenSearch index bootstrapped:
   ```bash
   uv run python -m src.migrations.init_db upgrade
   uv run python -m src.migrations.migrate_ddxplus_index upgrade
   ```
3. Run from project root or `notebooks/` with the project venv:
   ```bash
   cd disease-diagnosis-rag-system
   uv sync
   ```

First BGE / reranker calls download models into `models/` (lazy singleton).

In [17]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "services" / "rag").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "pyproject.toml").exists(), (
    "Run this notebook from the repo root or notebooks/ folder"
)

os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

Project root: /home/mitu/projects/disease-diagnosis-rag-system


In [18]:
import json

from src.db.vector_db.opensearch import get_opensearch_client
from src.services.inference.embeddings.service import TextEmbeddingService
from src.services.inference.reranker.service import RerankerService
from src.services.rag import RAGService, Retriever
from src.services.rag.ingest import Ingestion
from src.services.rag.preprocess import PreprocessPipeline
from src.services.rag.schemas import (
    Bm25RetrieveRequest,
    DiseaseDocument,
    HybridRetrieveRequest,
    RetrievalMode,
    RetrieveExperimentRequest,
    RetrieveResult,
    VectorRetrieveRequest,
)
from src.settings import settings

## Configuration

Non-secret defaults from `src/settings.py` and `.env`. Paths resolve against the project root.

In [19]:
print("Index alias:", settings.RETRIEVE_INDEX_ALIAS)
print("Search pipeline:", settings.CURRENT_SEARCH_PIPELINE)
print("Retrieve top_k:", settings.RETRIEVE_TOP_K)
print("Rerank top_k:", settings.RERANK_TOP_K)
print("Embedding model path:", settings.embedding_model_path)
print("Reranker model path:", settings.reranker_model_path)
print("OpenSearch host:", settings.OPENSEARCH_HOST)

Index alias: diseases
Search pipeline: hybrid-rrf
Retrieve top_k: 20
Rerank top_k: 5
Embedding model path: /home/mitu/projects/disease-diagnosis-rag-system/models/bge-small-en-v1.5
Reranker model path: /home/mitu/projects/disease-diagnosis-rag-system/models/bge-reranker-base
OpenSearch host: os-150fb08b-aio2026-conquer.g.aivencloud.com


## Step 1 — Ingest sample documents

`DiseaseDocument` holds source fields; `Ingestion` normalizes symptoms, embeds `embed_text`, derives `keyword_text`, and bulk-upserts into the `diseases` alias.

Skip this step if your index is already populated (e.g. full DDXPlus load).

In [20]:
REINGEST = False
records = [
    DiseaseDocument(
        doc_id="demo-1",
        disease="Influenza",
        symptoms=["fever", "cough"],
        severity=2,
        source="ddxplus",
        description="Example disease for notebook smoke test.",
    )
]

embed_service = TextEmbeddingService()
if REINGEST:
    ingestion = Ingestion(embed_service)
    count = ingestion.ingest(records)
else:
    count = 0
print(f"Ingested {count} record(s) into alias '{settings.RETRIEVE_INDEX_ALIAS}'")

Ingested 0 record(s) into alias 'diseases'


## Step 2 — Query preprocessing

Normalize symptom tokens and expand synonyms before search (`tired` → `fatigue`).

In [21]:
raw_query = "I am tired, have fever and cough"
preprocess = PreprocessPipeline()
search_query = preprocess.preprocess_query(raw_query)

print("Raw:", raw_query)
print("Preprocessed:", search_query)

Raw: I am tired, have fever and cough
Preprocessed: i am fatigue have fever and cough


## Step 3 — Initialize retrieval services

Reuses the embedding service from ingest. First reranker call loads `bge-reranker-base`.

- **`Retriever`** — BM25 / k-NN / hybrid APIs and composable `rerank()`
- **`RAGService`** — production path: hybrid retrieve (top 20) → rerank (top 5)

In [23]:
client = get_opensearch_client()
rerank_service = RerankerService()
preprocess = PreprocessPipeline()

retriever = Retriever(
    client=client,
    embed_service=embed_service,
    preprocess=preprocess,
)
retriever_with_rerank = Retriever(
    client=client,
    embed_service=embed_service,
    preprocess=preprocess,
    rerank_service=rerank_service,
)

QUERY = "fever cough fatigue"
TOP_K = 5

In [25]:
def show_hits(result: RetrieveResult, limit: int = 10) -> None:
    print(f"took_ms={result.took_ms}  hits={len(result.hits)}")
    for hit in result.hits[:limit]:
        score = f"{hit.score:.4f}" if hit.score is not None else "n/a"
        print(
            f"{hit.rank:>2}. {hit.disease:<24} "
            f"score={score}  symptoms={hit.symptoms}"
        )

## Step 4a — BM25 (lexical)

`match` on `keyword_text` with `operator: or`.

In [27]:
bm25_result = retriever.search_bm25(
    Bm25RetrieveRequest(query=QUERY, top_k=TOP_K)
)
print("=== BM25 ===")
show_hits(bm25_result)

=== BM25 ===
took_ms=2  hits=5
 1. Chagas                   score=1.5245  symptoms=['fever', 'new fatigue, generalized and vague discomfort, diffuse muscle aches or a change in your general well-being related to your consultation today', 'unintentionally losing weight', 'nauseous', 'diarrhea or an increase in stool frequency', 'swollen or painful lymph nodes', 'pain present', 'pain related to consultation', 'pain radiation', 'pain character', 'pain onset speed', 'pain intensity', 'pain location', 'swelling location', 'swelling in one or more areas of your body', 'shortness of breath or difficulty breathing in a significant way']
 2. Pneumonia                score=1.5153  symptoms=['pain that is increased when you breathe in deeply', 'pain present', 'pain related to consultation', 'pain radiation', 'pain character', 'pain onset speed', 'pain intensity', 'pain location', 'coughing up blood', 'rash location', 'lesions, redness or problems on your skin that you believe are related to the c

## Step 4b — k-NN (semantic)

BGE query embedding (384-dim, cosine) on the `embedding` field.

In [28]:
vector_result = retriever.search_vector(
    VectorRetrieveRequest(query=QUERY, top_k=TOP_K)
)
print("=== k-NN ===")
show_hits(vector_result)

=== k-NN ===
took_ms=2  hits=4
 1. Acute laryngitis         score=0.8629  symptoms=['the tone of your voice has become deeper, softer or hoarse', 'fever', 'cough', 'pain present', 'pain related to consultation', 'pain radiation', 'pain character', 'pain onset speed', 'pain intensity', 'pain location']
 2. Viral pharyngitis        score=0.8566  symptoms=['pain present', 'pain related to consultation', 'pain radiation', 'pain character', 'pain onset speed', 'pain intensity', 'pain location', 'nasal congestion / runny nose', 'fever', 'cough', 'coughing up blood']
 3. Bronchitis               score=0.8556  symptoms=['are your symptoms more prominent at night', 'cough', 'sore throat', 'nasal congestion / runny nose', 'cough that produces colored or more abundant sputum than usual', 'pain present', 'pain related to consultation', 'pain radiation', 'pain character', 'pain onset speed', 'pain intensity', 'pain location', 'wheezing sound when you exhale', 'fever', 'shortness of breath or diffic

## Step 4c — Hybrid + RRF (MVP default)

Single OpenSearch `hybrid` query fused server-side via the `hybrid-rrf` pipeline.

In [29]:
hybrid_result = retriever.search_hybrid(
    HybridRetrieveRequest(query=QUERY, top_k=TOP_K)
)
print("=== Hybrid + RRF ===")
show_hits(hybrid_result)

=== Hybrid + RRF ===
took_ms=4  hits=5
 1. Pneumonia                score=0.0318  symptoms=['pain that is increased when you breathe in deeply', 'pain present', 'pain related to consultation', 'pain radiation', 'pain character', 'pain onset speed', 'pain intensity', 'pain location', 'coughing up blood', 'rash location', 'lesions, redness or problems on your skin that you believe are related to the condition you are consulting for', 'rash color', 'rash pain intensity', 'rash swollen', 'itching severity', 'lesion larger than 1cm', 'lesions peel off', 'new fatigue, generalized and vague discomfort, diffuse muscle aches or a change in your general well-being related to your consultation today', 'loss of appetite', 'are your symptoms more prominent at night', 'fever', 'cough that produces colored or more abundant sputum than usual', 'cough', 'shortness of breath or difficulty breathing in a significant way', 'chills or shivers', 'diffuse muscle pain', 'nasal congestion / runny nose']
 2. Br

## Step 5 — Compare all retrieval modes

Same query through BM25, k-NN, and hybrid side by side.

In [14]:
comparison = retriever.run_experiment(
    RetrieveExperimentRequest(query=QUERY, top_k=TOP_K)
)

for mode, mode_result in comparison.results.items():
    print(f"\n=== {mode.value.upper()} (total_hits={mode_result.total_hits}) ===")
    show_hits(mode_result, limit=3)


=== BM25 (total_hits=28) ===
took_ms=2  hits=5
 1. Chagas                   score=1.5245  symptoms=['fever', 'new fatigue, generalized and vague discomfort, diffuse muscle aches or a change in your general well-being related to your consultation today', 'unintentionally losing weight', 'nauseous', 'diarrhea or an increase in stool frequency', 'swollen or painful lymph nodes', 'pain present', 'pain related to consultation', 'pain radiation', 'pain character', 'pain onset speed', 'pain intensity', 'pain location', 'swelling location', 'swelling in one or more areas of your body', 'shortness of breath or difficulty breathing in a significant way']
 2. Pneumonia                score=1.5153  symptoms=['pain that is increased when you breathe in deeply', 'pain present', 'pain related to consultation', 'pain radiation', 'pain character', 'pain onset speed', 'pain intensity', 'pain location', 'coughing up blood', 'rash location', 'lesions, redness or problems on your skin that you believe are

## Step 6 — Retrieve + rerank (production path)

Hybrid search returns up to `RETRIEVE_TOP_K` (20) hits. Cross-encoder reranking keeps `RERANK_TOP_K` (5).

Each hit exposes `passage_text` — symptom-first text for the reranker. Query preprocessing runs via the injected `PreprocessPipeline`.

In [15]:
retrieved = retriever_with_rerank.retrieve(QUERY)
print(f"Before rerank: {len(retrieved.hits)} hits")
if retrieved.hits:
    print("Sample passage_text:", retrieved.hits[0].passage_text[:120], "...")

reranked = retriever_with_rerank.rerank(
    QUERY, retrieved, top_k=settings.RERANK_TOP_K
)
print("\n=== After rerank (cross-encoder scores) ===")
show_hits(reranked)

rag = RAGService(
    embed_service=embed_service,
    rerank_service=rerank_service,
)
result = rag.query(QUERY)
print("\n=== RAGService.query() ===")
show_hits(result)

Before rerank: 20 hits
Sample passage_text: Disease: Pneumonia. Symptoms: pain that is increased when you breathe in deeply, pain present, pain related to consultat ...
2026-07-01T16:23:38.140474Z [info     ] No device provided, using cuda:0 [sentence_transformers.base.model]
2026-07-01T16:23:38.140976Z [info     ] No modules.json found for /home/mitu/projects/disease-diagnosis-rag-system/models/bge-reranker-base, initializing a new CrossEncoder model. [sentence_transformers.base.model]


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.60it/s]



=== After rerank (cross-encoder scores) ===
took_ms=13  hits=5
 1. Pneumonia                score=0.9228  symptoms=['pain that is increased when you breathe in deeply', 'pain present', 'pain related to consultation', 'pain radiation', 'pain character', 'pain onset speed', 'pain intensity', 'pain location', 'coughing up blood', 'rash location', 'lesions, redness or problems on your skin that you believe are related to the condition you are consulting for', 'rash color', 'rash pain intensity', 'rash swollen', 'itching severity', 'lesion larger than 1cm', 'lesions peel off', 'new fatigue, generalized and vague discomfort, diffuse muscle aches or a change in your general well-being related to your consultation today', 'loss of appetite', 'are your symptoms more prominent at night', 'fever', 'cough that produces colored or more abundant sputum than usual', 'cough', 'shortness of breath or difficulty breathing in a significant way', 'chills or shivers', 'diffuse muscle pain', 'nasal congest

Batches: 100%|██████████| 1/1 [00:00<00:00, 91.97it/s]



=== RAGService.query() ===
took_ms=11  hits=5
 1. Pneumonia                score=0.9228  symptoms=['pain that is increased when you breathe in deeply', 'pain present', 'pain related to consultation', 'pain radiation', 'pain character', 'pain onset speed', 'pain intensity', 'pain location', 'coughing up blood', 'rash location', 'lesions, redness or problems on your skin that you believe are related to the condition you are consulting for', 'rash color', 'rash pain intensity', 'rash swollen', 'itching severity', 'lesion larger than 1cm', 'lesions peel off', 'new fatigue, generalized and vague discomfort, diffuse muscle aches or a change in your general well-being related to your consultation today', 'loss of appetite', 'are your symptoms more prominent at night', 'fever', 'cough that produces colored or more abundant sputum than usual', 'cough', 'shortness of breath or difficulty breathing in a significant way', 'chills or shivers', 'diffuse muscle pain', 'nasal congestion / runny nose'

## Optional — Inspect OpenSearch request body

Enable debug bodies on the experiment request to see the Query DSL sent to OpenSearch.

In [16]:
debug = retriever.run_experiment(
    RetrieveExperimentRequest(
        query=QUERY,
        top_k=TOP_K,
        include_opensearch_body=True,
    )
)

hybrid_debug = debug.results[RetrievalMode.HYBRID]
print(json.dumps(hybrid_debug.opensearch_body, indent=2)[:2000])

{
  "size": 5,
  "_source": [
    "doc_id",
    "disease",
    "symptoms",
    "antecedents",
    "severity",
    "description",
    "source"
  ],
  "query": {
    "hybrid": {
      "queries": [
        {
          "match": {
            "keyword_text": {
              "query": "fever cough fatigue",
              "operator": "or"
            }
          }
        },
        {
          "knn": {
            "embedding": {
              "vector": [
                -0.04329468309879303,
                -0.021098092198371887,
                -0.012204569764435291,
                0.023148415610194206,
                0.07119158655405045,
                -0.024460140615701675,
                0.06295740604400635,
                -0.0164052564650774,
                -0.0617455318570137,
                -0.0381803885102272,
                -0.06582057476043701,
                -0.02865307405591011,
                0.01843469962477684,
                0.031149817630648613,
                -0.

## Next steps

- **Full ingest** — load `data/kb/kb_ddxplus.json` via `RAGService.ingest()` (see `docs/ddxplus-index-mapping.md`)
- **Generate** — Qwen3 8B with reranked context
- **API** — FastAPI endpoint wrapping `RAGService.query()`